# Ngày 25 · Tối ưu và regularization

- Cài SGD có momentum và Adam đúng như PyTorch
- Viết dropout đảo (inverted dropout) và hiểu vì sao tắt nó khi đánh giá
- Dừng sớm theo validation với tham số patience

**Đọc:**
- Goodfellow, Bengio & Courville (2016), Deep Learning, §8.3.2 Momentum và §8.5.3 Adam; §7.8 Early Stopping; §7.12 Dropout
- Kingma & Ba (2015), Adam: A Method for Stochastic Optimization, ICLR
- Srivastava et al. (2014), Dropout: A Simple Way to Prevent Neural Networks from Overfitting, JMLR 15, 1929–1958

In [ ]:
import json, math, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd

DATA = next((p for p in (Path('data'), Path('../data')) if p.is_dir()), Path('data'))

def _check(name, fn):
    try:
        ok, detail = fn()
    except NotImplementedError:
        ok, detail = False, 'Chưa cài đặt (NotImplementedError)'
    except Exception as error:
        ok, detail = False, f'{type(error).__name__}: {error}'
    return {'name': name, 'ok': bool(ok), 'detail': str(detail)}

def _report(ex_id, *checks):
    print('__PY30__' + json.dumps({'id': ex_id, 'checks': list(checks)}, ensure_ascii=False))
try:
    import torch
    from torch import nn
    import torch.nn.functional as F
except ImportError as error:
    raise ImportError('Kernel này chưa có PyTorch. Cài bằng Python của Jupyter: python -m pip install -r requirements-py30.txt') from error
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
# load_digits (UCI Optical Recognition of Handwritten Digits): 1797 ảnh 8×8, chia 80/20 phân tầng, seed 0.
_Xd, _yd = load_digits(return_X_y=True)
_itr, _iva = train_test_split(np.arange(len(_yd)), test_size=0.2, random_state=0, stratify=_yd)
X_tr = torch.tensor(_Xd[_itr] / 16, dtype=torch.float32); y_tr = torch.tensor(_yd[_itr])
X_va = torch.tensor(_Xd[_iva] / 16, dtype=torch.float32); y_va = torch.tensor(_yd[_iva])
def _loader(X, y, batch_size, seed):
    return torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X, y), batch_size=batch_size, shuffle=True,
                                       generator=torch.Generator().manual_seed(seed))
def _train_epoch(model, loader, loss_fn, optimizer):
    model.train(); total, n = 0.0, 0
    for xb, yb in loader:
        optimizer.zero_grad(); loss = loss_fn(model(xb), yb); loss.backward(); optimizer.step()
        total += loss.item() * len(yb); n += len(yb)
    return total / n
def _accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        return float((model(X).argmax(1) == y).float().mean())
_A25 = np.array([10., 1., 3.])
def _torch_run(make_opt, steps):
    w = torch.tensor([1., 1., -2.], requires_grad=True); opt = make_opt([w]); out = []
    for _ in range(steps):
        opt.zero_grad(); (0.5 * (torch.tensor(_A25, dtype=torch.float32) * w ** 2).sum()).backward(); opt.step(); out.append(w.detach().numpy().copy())
    return np.array(out)


## SGD có momentum và Adam

Gợi ý: Quy ước PyTorch: v ← momentum·v + g; w ← w − lr·v. Adam: m ← β₁m + (1−β₁)g, v ← β₂v + (1−β₂)g², m̂ = m/(1−β₁ᵗ), v̂ = v/(1−β₂ᵗ), w ← w − lr·m̂/(√v̂ + ε). t bắt đầu từ 1.

In [ ]:
def sgd_momentum_step(w, g, v, lr, momentum):
    # Trả (w_mới, v_mới). Không sửa mảng đầu vào tại chỗ.
    raise NotImplementedError

def adam_step(w, g, m, v, t, lr, beta1=0.9, beta2=0.999, eps=1e-8):
    # Trả (w_mới, m_mới, v_mới).
    raise NotImplementedError


In [ ]:
def _mine(kind, steps):
    w = np.array([1., 1., -2.]); s1 = np.zeros(3); s2 = np.zeros(3); out = []
    for t in range(1, steps + 1):
        g = _A25 * w
        if kind == 'sgd':
            w, s1 = sgd_momentum_step(w, g, s1, 0.05, 0.9)
        else:
            w, s1, s2 = adam_step(w, g, s1, s2, t, 0.1)
        out.append(w.copy())
    return np.array(out)
def _no_inplace():
    w, g, v = np.ones(3), np.ones(3), np.zeros(3); sgd_momentum_step(w, g, v, .1, .9)
    return np.all(w == 1) and np.all(v == 0)
_report('d25-optim',
    _check('SGD momentum khớp torch.optim.SGD qua 20 bước', lambda: (np.allclose(_mine('sgd', 20), _torch_run(lambda p: torch.optim.SGD(p, lr=0.05, momentum=0.9), 20), atol=1e-5), 'lr = 0,05, momentum = 0,9.')),
    _check('Adam khớp torch.optim.Adam qua 20 bước', lambda: (np.allclose(_mine('adam', 20), _torch_run(lambda p: torch.optim.Adam(p, lr=0.1), 20), atol=1e-5), 'lr = 0,1, β = (0,9; 0,999).')),
    _check('Không sửa mảng đầu vào tại chỗ', lambda: (_no_inplace(), 'Trả mảng mới.')),
)


## Dropout đảo và dừng sớm

Gợi ý: dropout: khi training, mask = rng.random(x.shape) >= p, trả x·mask/(1 − p); khi đánh giá trả x. early_stopping: theo dõi loss nhỏ nhất; nếu không giảm trong patience epoch liên tiếp thì dừng ở epoch đó.

In [ ]:
def dropout(x, p, training, rng):
    raise NotImplementedError

def early_stopping(val_losses, patience):
    # Trả (best_epoch, stop_epoch), đếm từ 0. Chạy hết danh sách mà không dừng thì stop_epoch = len − 1.
    raise NotImplementedError


In [ ]:
_losses25 = [1.0, .8, .7, .72, .69, .71, .73, .74, .6]
_report('d25-reg',
    _check('Giữ nguyên kỳ vọng và tỉ lệ bỏ đúng p', lambda: ((lambda z: abs(z.mean() - 1) < .02 and abs((z == 0).mean() - .3) < .01)(dropout(np.ones(200000), .3, True, np.random.default_rng(0))), 'p = 0,3 trên 200.000 phần tử.')),
    _check('Khi đánh giá trả nguyên x', lambda: (np.array_equal(dropout(np.arange(5.), .5, False, np.random.default_rng(0)), np.arange(5.)), 'training = False.')),
    _check('Dừng sớm đúng epoch', lambda: (early_stopping(_losses25, 3) == (4, 7) and early_stopping(_losses25, 10) == (8, 8), 'patience = 3: tốt nhất epoch 4, dừng ở epoch 7 (bỏ lỡ 0,6 ở epoch 8).')),
)


## Dự án

Trong notebook cloud, huấn luyện cùng một MLP trên Fashion-MNIST với SGD, SGD + momentum 0,9 và Adam (cùng số epoch). Vẽ loss validation của ba cách trên một hình. Thêm Dropout(0,3) và dừng sớm với patience = 3, ghi epoch dừng và accuracy test.